# Predicted Epitopes of Trypanosoma cruzi Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset: Predicted Epitopes of _Trypanosoma cruzi_ based on Phage Display Data, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (`https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json`).

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We print a summary of the available record sets and their fields as described by the dataset's metadata, referencing all entities by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record sets:\n")
all_record_set_ids = []
record_set_fields = {}
for rset in record_sets:
    print(f"RecordSet: {rset.id}")
    all_record_set_ids.append(rset.id)
    fields_ids = [field.id for field in getattr(rset, 'fields', [])]
    record_set_fields[rset.id] = fields_ids
    print("  Fields:")
    for field in getattr(rset, 'fields', []):
        print(f"    - {field.id}")
    print()
if not record_sets:
    print("No record sets found. Please ensure this dataset has record sets defined in the Croissant schema.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.

Below, we extract all available record sets, and show field IDs for each. We preview the first record set if at least one is available.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}

for rs_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Show columns for the first available record set (if exists)
if all_record_set_ids:
    first_rs_id = all_record_set_ids[0]
    print(f"\nColumns for first record set: {first_rs_id}\n{dataframes.get(first_rs_id, pd.DataFrame()).columns.tolist()}")
    display(dataframes.get(first_rs_id, pd.DataFrame()).head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this section, we use the first record set (if available), and reference numeric and group fields by their `@id`. You may edit `numeric_field_id` and `group_field_id` to point to fields of interest from the data overview above.

In [ ]:
import numpy as np

# Choose a record set to explore
if all_record_set_ids:
    record_set_id = all_record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Exploring record set: {record_set_id}")

    # Choose a numeric field by @id (replace with valid @id as appropriate)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a group field (@id), use a non-numeric field if present
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the selected record set. EDA is skipped.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below we visualize the numeric field chosen above, and its grouping (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the distribution and grouped stats if possible
if all_record_set_ids and 'numeric_field_id' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id} (in {record_set_id})")

    if 'grouped_df' in locals() and len(grouped_df) > 0:
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=axes[1])
        axes[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")
    else:
        axes[1].remove()

    plt.tight_layout()
    plt.show()
else:
    print("Nothing to plot. Please ensure numeric and group fields exist and the record set is not empty.")

## 6. Conclusion
This notebook demonstrated the use of the `mlcroissant` library for structured exploration of a Croissant-described dataset, referencing all entities by their `@id`. We:
- Loaded dataset metadata and reviewed available record sets and fields (with `@id`).
- Extracted and analyzed data from each record set (referenced by `@id`).
- Conducted basic EDA including filtering, normalization, grouping by key fields, and visualization.

You can further customize field selection by editing field `@id`s in the notebook, and adapt the workflow for domain-specific downstream analyses.